In [4]:
from ddgs import DDGS

documents=[]

# Step 1: Define queries
queries_for_news_Source_tag = [
    "Lufthansa financial results",
    "Lufthansa news 2026",
    "Lufthansa strike",
    "Lufthansa fleet order Airbus",
    "Lufthansa competitors",
    "Lufthansa CEO strategy",
    "Lufthansa profit loss quarter",
]
queries_for_Reddit_source_tag = [
    "Lufthansa site:reddit.com",
    "Lufthansa review site:reddit.com",
    "Lufthansa delay site:reddit.com",
    "Lufthansa experience site:reddit.com",
]

queries_for_competitor_tag = [
    "Air France KLM financial results",
    "Air France KLM new routes fleet",
    "IAG British Airways strategy",
    "Ryanair expansion profit",
    "easyJet growth news",
    "Emirates fleet order",
    "Turkish Airlines expansion",
]

queries_for_company_tag = [
    "Lufthansa site:lufthansagroup.com",
    "Lufthansa press release site:lufthansagroup.com",
    "Lufthansa annual report site:lufthansagroup.com",
]

# Step 2: Search recent news using DuckDuckGo / DDGS
with DDGS() as ddgs:

    for q in queries_for_news_Source_tag:
        for r in ddgs.text(q, max_results=20):
            documents.append({
            "text":   r["title"] + ". " + r["body"],
            "url":    r["href"],     
            "source": "news",
            })

    for q in queries_for_Reddit_source_tag:
        for r in ddgs.text(q, max_results=20):
            documents.append({
            "text":   r["title"] + ". " + r["body"],
            "url":    r["href"],     
            "source": "reddit",
            })

    for q in queries_for_competitor_tag:
        for r in ddgs.text(q, max_results=20):
            documents.append({
                "text":   r["title"] + ". " + r["body"],
                "url":    r["href"],
                "source": "competitor",   
            })

    for q in queries_for_company_tag:
        for r in ddgs.text(q, max_results=20):
            documents.append({
                "text":   r["title"] + ". " + r["body"],
                "url":    r["href"],
                "source": "company",   
            })

# Display length of Documents
print(len(documents))

402


### Data Cleaning

In [5]:
import re

#removing duplicates
unique_docs = list({d["url"]: d for d in documents}.values())

def clean_text(text):
    text = re.sub(r"\s+", " ", text)        # normalize whitespace
    text = re.sub(r"\.{2,}", ".", text)     # collapse "...."
    return text.strip()

clean_docs, seen_text = [], set()
for d in unique_docs:
    t = clean_text(d["text"])
    if len(t) <= 30:                            # too short / empty
        continue
    if len(set(t.lower().split())) < 10:        # low-info / navigation page
        continue
    if t in seen_text:                          # text duplicate (URL-dedup missed)
        continue
    seen_text.add(t)
    d["text"] = t
    clean_docs.append(d)

print("After URL-dedup:", len(unique_docs), "| After cleaning:", len(clean_docs))



After URL-dedup: 377 | After cleaning: 341


In [6]:
# save the CLEAN data
import json
json.dump(clean_docs, open("data/lufthansa_data.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saved", len(clean_docs), "clean docs")

Saved 341 clean docs
